# Лабораторная работа 11

Датасет fer2013. Изображения лиц с 7 классами


# Текстовый трансформер из 10 лабы

In [90]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

In [91]:
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [92]:
length = 256
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=length
    )

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [93]:
print(dataset["train"][0]["text"])
print(dataset["train"][0]["input_ids"])
print(dataset["train"][0]["attention_mask"])
print(dataset["train"][0]["label"])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [94]:
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(dataset["train"], batch_size=32, shuffle=True)
test_loader = DataLoader(dataset["test"], batch_size=32)

In [95]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv = nn.Linear(d_model, d_model * 3)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
          mask = mask.unsqueeze(1).unsqueeze(2)
          scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)

        return self.out(out)

In [96]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [97]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dim_ff=2048, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.attn(x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

In [98]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads) for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [99]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional = PositionalEncoding(d_model)
        self.encoder = TransformerEncoder(num_layers, d_model, num_heads)
        self.cls = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x = self.positional(x)
        x = self.encoder(x, attention_mask)
        pooled = x.mean(dim=1)
        return self.cls(pooled)

In [100]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [101]:
print(tokenizer.vocab_size)
print(len(tokenizer))

30522
30522


In [102]:
model = TransformerClassifier(
    vocab_size=len(tokenizer),
    d_model=256,
    num_heads=8,
    num_layers=4,
    num_classes=2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

In [103]:
for epoch in range(5):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )
        loss = criterion(logits, batch["label"].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

Epoch 1: loss=0.5886
Epoch 2: loss=0.4141
Epoch 3: loss=0.3388
Epoch 4: loss=0.2783
Epoch 5: loss=0.2298


# VisualTransformer


In [104]:
!gdown -q 1QtDJzXGdwU8WdmZHXPNzTJjr6xzHZrDW
!unzip -q /content/archive.zip

replace test/angry/PrivateTest_10131363.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [105]:
import torch.nn.functional as F
from torch.utils.data import Dataset

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from PIL import Image

In [106]:
class FERFolderDataset(Dataset):
    def __init__(self, root_dir, image_size=48):
        self.root_dir = root_dir
        self.image_size = image_size

        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.samples = []
        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                self.samples.append(
                    (os.path.join(cls_dir, fname), self.class_to_idx[cls])
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("L")
        img = img.resize((self.image_size, self.image_size))
        img = np.array(img, dtype=np.float32) / 255.0

        img = torch.tensor(img).unsqueeze(0)
        return img, label

train_ds = FERFolderDataset("train")
test_ds  = FERFolderDataset("test")

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64)


In [107]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=48, patch_size=6, in_chans=1, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


In [108]:
class ViTEmbedding(nn.Module):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

    def forward(self, x):
        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        return x + self.pos_embed


In [109]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=48,
        patch_size=6,
        embed_dim=256,
        num_heads=8,
        num_layers=4,
        num_classes=7
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(
            img_size, patch_size, 1, embed_dim
        )
        self.embed = ViTEmbedding(
            self.patch_embed.num_patches,
            embed_dim
        )
        self.encoder = TransformerEncoder(
            num_layers, embed_dim, num_heads
        )
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.embed(x)
        x = self.encoder(x)
        cls = x[:, 0]
        return self.head(cls)


In [110]:
vit = VisionTransformer().to(device)
optimizer = torch.optim.Adam(vit.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

In [111]:
value = next(iter(train_loader))

In [112]:
print(value[0].shape)
print(value[1].shape)

torch.Size([64, 1, 48, 48])
torch.Size([64])


In [113]:
for epoch in range(10):
    vit.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        logits = vit(batch[0].to(device))
        loss = criterion(logits, batch[1].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")


Epoch 1: loss=1.8286
Epoch 2: loss=1.7749
Epoch 3: loss=1.7127
Epoch 4: loss=1.6503
Epoch 5: loss=1.5970
Epoch 6: loss=1.5614
Epoch 7: loss=1.5163
Epoch 8: loss=1.4763
Epoch 9: loss=1.4422
Epoch 10: loss=1.4043


In [114]:
from sklearn.metrics import accuracy_score

vit.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in test_loader:
        logits = vit(batch[0].to(device))
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(batch[1].numpy())

print("Accuracy:", accuracy_score(labels, preds))


Accuracy: 0.4271384786848704


In [115]:
emotion_texts = [
    "an angry person",
    "a disgusted person",
    "a fearful person",
    "a happy person",
    "a sad person",
    "a surprised person",
    "a neutral person"
]

In [116]:
class TextEncoder(nn.Module):
    def __init__(self, transformer):
        super().__init__()
        self.transformer = transformer

    def forward(self, input_ids, attention_mask):
        x = self.transformer.embedding(input_ids)
        x = self.transformer.positional(x)
        x = self.transformer.encoder(x, attention_mask)
        return x.mean(dim=1)


In [117]:
class ImageEncoder(nn.Module):
    def __init__(self, vit):
        super().__init__()
        self.vit = vit

    def forward(self, x):
        x = self.vit.patch_embed(x)
        x = self.vit.embed(x)
        x = self.vit.encoder(x)
        return x[:, 0]


In [118]:
max_len = 16
text_inputs = tokenizer(
    emotion_texts,
    padding="max_length",
    truncation=True,
    max_length=max_len,
    return_tensors="pt"
)

In [119]:
print(text_inputs["input_ids"])
print(text_inputs["attention_mask"])

tensor([[  101,  2019,  4854,  2711,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1037, 17733,  2711,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1037, 19725,  2711,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1037,  3407,  2711,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1037,  6517,  2711,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1037,  4527,  2711,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1037,  8699,  2711,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0]])
tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 

In [120]:
text_encoder = TextEncoder(transformer=model).to(device)
image_encoder = ImageEncoder(vit=vit).to(device)

text_encoder.train()
image_encoder.train()

ImageEncoder(
  (vit): VisionTransformer(
    (patch_embed): PatchEmbedding(
      (proj): Conv2d(1, 256, kernel_size=(6, 6), stride=(6, 6))
    )
    (embed): ViTEmbedding()
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (attn): MultiHeadAttention(
            (qkv): Linear(in_features=256, out_features=768, bias=True)
            (out): Linear(in_features=256, out_features=256, bias=True)
          )
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (ff): Sequential(
            (0): Linear(in_features=256, out_features=2048, bias=True)
            (1): ReLU()
            (2): Linear(in_features=2048, out_features=256, bias=True)
          )
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (head): Linear(in_features=256, out_features=7, bias=True)
  )
)

In [121]:
import torch.nn.functional as F

temperature = 0.07

def contrastive_loss(image_emb, text_emb, labels):
  image_emb = F.normalize(image_emb, dim=1)
  text_emb = F.normalize(text_emb, dim=1)

  logits = image_emb @ text_emb.T / temperature
  return torch.nn.CrossEntropyLoss()(logits, labels)


In [122]:
optimizer = torch.optim.Adam(
    list(image_encoder.parameters()) + list(text_encoder.parameters()),
    lr=3e-4
)


In [131]:
num_epochs = 5

text_encoder.train()
image_encoder.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        images = batch[0].to(device)
        labels = batch[1].to(device)

        image_emb = image_encoder(images)

        text_emb = text_encoder(
            input_ids=text_inputs["input_ids"].to(device),
            attention_mask=text_inputs["attention_mask"].to(device)
        )

        loss = contrastive_loss(image_emb, text_emb, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, contrastive loss: {total_loss/len(train_loader):.4f}")


Epoch 1, contrastive loss: 0.5206
Epoch 2, contrastive loss: 0.4864
Epoch 3, contrastive loss: 0.4408
Epoch 4, contrastive loss: 0.3975
Epoch 5, contrastive loss: 0.3820


In [135]:
image_encoder.eval()
text_encoder.eval()

sample_images = next(iter(test_loader))[0][:1].to(device)
sample_labels = next(iter(test_loader))[1][:1]

with torch.no_grad():
    image_emb = image_encoder(sample_images)
    text_emb = text_encoder(
        input_ids=text_inputs["input_ids"].to(device),
        attention_mask=text_inputs["attention_mask"].to(device)
    )

    image_emb_norm = F.normalize(image_emb, dim=1)
    text_emb_norm = F.normalize(text_emb, dim=1)
    similarity = F.cosine_similarity(
      image_emb_norm.unsqueeze(1),
      text_emb_norm.unsqueeze(0),
      dim=2
)

print("Similarity matrix (image x text):")
print(similarity)


Similarity matrix (image x text):
tensor([[ 0.3655, -0.1676, -0.1527, -0.0199, -0.2082,  0.0589, -0.1127]],
       device='cuda:0')


In [129]:
sample_labels

tensor([0, 0, 0, 0, 0])

Сверху вектор косинусной близости, изображение angry. Видно что эмбеддинг "an angry person", в большей мере совпадает с эмбеддингом angry изображения лица, чем остальные. 0.365